# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [3]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [4]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [5]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un tip de inteligență artificială antrenată pe volume masive de text, care îi permite să înțeleagă, să genereze și să manipuleze limbajul uman într-un mod fluent și coerent. Aceste modele pot realiza o gamă largă de sarcini lingvistice, de la răspunsuri la întrebări și traduceri, până la scrierea creativă și rezumarea de texte.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenat pe cantități masive de text și cod, capabil să înțeleagă și să genereze limbaj uman într-un mod fluent și coerent. Aceste modele pot efectua o gamă largă de sarcini lingvistice, de la răspunsuri la întrebări și traduceri, până la crearea de conținut original și rezumarea textelor.


In [6]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [8]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, de ce au fost anulate alegerile în 2024 în România.
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
Alegerile din 2024 în România au fost anulate din cauza unei decizii a Curții Constituționale care a declarat neconstituțională legea privind organizarea alegerilor locale. Această decizie a necesitat modificări legislative, determinând amânarea scrutinului.

--- Gemini 2.5 Flash ---
Alegerile din 2024 în România nu au fost anulate. Dimpotrivă, alegerile locale și europarlamentare au avut loc pe 9 iunie, iar cele prezidențiale și parlamentare sunt programate pentru toamnă.

--- OpenRouter Free ---
Alergerile au fost anulate de Curtea Constituțională a României în 2024 datorită unor probleme legate de drepturile electorale și de transparența procesului electoral. Aceste decizii au fost luate pentru a asigura că alegerile viitoare sunt echitabile și respectă standardele de democrație.


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [9]:
SYSTEM = """
Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Oamenii l-au votat pe Călin Georgescu pentru că „ne ferește de război sau că dacă ne înțelegem cu Rusia, nu vine Rusia peste noi"

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Informativ
Emoție dominantă: Frică
Țintă principală: Alegătorii
Populism: da

--- Gemini 2.5 Flash ---
Ton: Explicativ, justificativ.
Emoție dominantă: Frică (de război), dorință de siguranță.
Țintă principală: Electoratul, opinia publică.
Populism: da

--- OpenRouter Free ---
Ton: Critic.  
Emoție dominantă: Skepticism.  
Țintă principală: Votarea pentru Georgescu motivată de pașețire față de război/Rusia.  
Populism: Da.


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [10]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [11]:
COMENTARIU = "Oamenii l-au votat pe Călin Georgescu pentru că „ne ferește de război sau că dacă ne înțelegem cu Rusia, nu vine Rusia peste noi"


SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'neutru', 'emotie_dominanta': 'neutru', 'tinta_principala': 'Călin Georgescu', 'populism': False, 'explicatie_scurta': 'Comentariul prezintă motivul votului pentru Călin Georgescu, legat de percepția că acesta ar putea proteja țara de război sau de agresiunea rusă.'}

--- Gemini 2.5 Flash ---
{'ton': 'neutru', 'emotie_dominanta': 'neutru', 'tinta_principala': 'Călin Georgescu și susținătorii săi', 'populism': True, 'explicatie_scurta': 'Comentariul raportează motive populiste (evitarea războiului prin înțelegere cu Rusia) pentru votul acordat lui Călin Georgescu.'}

--- OpenRouter Free ---
{'emotie_dominanta': 'frica', 'explicatie_scurta': 'Comentariul se bazează pe apelul la frică și pe o promisiune de securitate externă, oferind o soluție simplistă (înțelegerea cu Rusia) pentru o problemă geopolitică complexă.', 'populism': True, 'tinta_principala': 'siguranta_nationala', 'ton': 'pozitiv'}


## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [12]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru popor.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională poate însemna că procesul electoral nu a respectat prevederile legale, necesitând organizarea unor noi alegeri. Acest lucru poate duce la o perioadă de incertitudine politică și la o amânare a formării unui nou guvern sau a unui nou parlament.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională poate însemna că procesul electoral nu a respectat prevederile legale sau constituționale, necesitând organizarea unor noi alegeri. Acest lucru poate duce la o perioadă de incertitudine politică și la necesitatea repetării campaniei electorale și a votului.

temperature=1.2:
Anularea alegerilor de către Curtea Constituțională poate însemna pentru popor necesitatea organizării unor noi scrutinuri, care vor presupune un nou proces electoral complet. Acest lucru ar putea duce la o perioadă de incertitudine politică și la am

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da  | da  | da  |  nu | |
| OpenRouter Free |  parțial | da  | parțial |  nu | |

### Decizie
**Model principal ales:**  
**Model de rezervă:**  
**Temperature recomandată:**  
**De ce am ales acest model?**  
Scrieți 2-3 propoziții. Menționați calitatea răspunsului, stabilitatea și dacă modelul poate fi folosit pentru adnotarea comentariilor.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [ ]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales